# Stage 14: Experiment Sensitivity Framework

This notebook runs a compact parameter study on the existing portfolio research stack and summarizes which settings are robust.

## 1. Load Data

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

project_root = Path.cwd().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_pipeline import DataPreprocessor, YahooFinanceProvider
from src.experiments import (
    default_phase2d_config,
    generate_parameter_grid,
    run_experiment_grid,
    rank_experiments,
    summarize_by_parameter,
    compute_parameter_sensitivity,
    build_top_n_table,
)


In [ ]:
symbols = ["HDFCBANK.NS", "ICICIBANK.NS", "SBIN.NS", "TCS.NS", "INFY.NS", "RELIANCE.NS", "ITC.NS", "GOLDBEES.NS"]
provider = YahooFinanceProvider()
market_data = provider.get_market_data(symbols, "2020-01-01", "2025-01-01")
prices_df, quality_summary = DataPreprocessor.handle_missing_values(market_data.prices_df)
returns_df = DataPreprocessor.build_returns_risk_outputs(prices_df).returns_df
returns_df.tail()

## 2. Build Experiment Config

In [ ]:
config = default_phase2d_config()
config

## 3. Generate Parameter Grid

In [ ]:
parameter_grid = generate_parameter_grid(config)
parameter_grid.head()

## 4. Run Grid

In [ ]:
experiment_results_df = run_experiment_grid(returns_df, config=config, max_runs=24)
experiment_results_df.head()

## 5. Rank Experiments

In [ ]:
ranked_by_calmar = rank_experiments(experiment_results_df, objective="calmar")
ranked_by_calmar.head(10)

## 6. Summarize by Strategy

In [ ]:
summarize_by_parameter(experiment_results_df, parameter="strategy", metric="calmar")

## 7. Summarize by Covariance Method

In [ ]:
cov_summary = summarize_by_parameter(experiment_results_df, parameter="covariance_method", metric="calmar")
cov_summary

## 8. Summarize by Rebalance Mode

In [ ]:
summarize_by_parameter(experiment_results_df, parameter="rebalance_mode", metric="calmar")

## 9. Sensitivity Heatmaps

In [ ]:
heatmap_data = experiment_results_df[experiment_results_df["status"] == "success"].pivot_table(
    index="strategy",
    columns="covariance_method",
    values="calmar",
    aggfunc="mean",
)
px.imshow(heatmap_data, color_continuous_scale="Blues", aspect="auto", title="Calmar by Strategy and Covariance Method")

## 10. Top 10 Configurations

In [ ]:
build_top_n_table(experiment_results_df, metric="calmar", n=10)

## 11. Interpretation

In [ ]:
compute_parameter_sensitivity(experiment_results_df, metric="calmar")